In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import classification_report, confusion_matrix
from catboost import CatBoostClassifier, Pool
import lightgbm as lgb

In [2]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/playground-series-s6e7/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e7/train.csv
/kaggle/input/competitions/playground-series-s6e7/test.csv


In [3]:
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/test.csv')
cat_cols = ['diet_type', 'stress_level', 'sleep_quality',
            'physical_activity_level', 'smoking_alcohol', 'gender']
num_cols = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure',
            'step_count', 'exercise_duration', 'water_intake']

In [4]:
le = LabelEncoder()
y = le.fit_transform(train['health_condition'])


X = train[cat_cols + num_cols].copy()
X_test = test[cat_cols + num_cols].copy()

# LightGBM handles missing values natively — no imputation needed
for c in cat_cols:
    X[c] = X[c].astype('category')
    X_test[c] = pd.Categorical(X_test[c], categories=X[c].cat.categories)

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
oof = np.zeros((len(X), 3))
test_preds = np.zeros((len(X_test), 3))

params = {
    'objective': 'multiclass',
    'num_class': 3,
    'metric': 'multi_logloss',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'min_child_samples': 50,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'lambda_l1': 0.1,
    'lambda_l2': 0.1,
    'verbosity': -1,
    'seed': 42,
}

In [5]:
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]
    sample_weight = compute_sample_weight(class_weight='balanced', y=y_tr)
    val_sample_weight = compute_sample_weight(class_weight='balanced', y=y_val)

    dtrain = lgb.Dataset(X_tr, label=y_tr, weight=sample_weight, categorical_feature=cat_cols)
    dval = lgb.Dataset(X_val, label=y_val, weight=val_sample_weight, categorical_feature=cat_cols, reference=dtrain)

    model = lgb.train(
        params, dtrain,
        num_boost_round=2000,
        valid_sets=[dval],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
    )

    oof[val_idx] = model.predict(X_val, num_iteration=model.best_iteration)
    test_preds += model.predict(X_test, num_iteration=model.best_iteration) / n_splits

    fold_bal_acc = balanced_accuracy_score(y_val, oof[val_idx].argmax(axis=1))
    print(f"Fold {fold} balanced acc: {fold_bal_acc:.5f}, best_iter: {model.best_iteration}")

Training until validation scores don't improve for 50 rounds
[100]	valid_0's multi_logloss: 0.175266
[200]	valid_0's multi_logloss: 0.171865
Early stopping, best iteration is:
[217]	valid_0's multi_logloss: 0.171771
Fold 0 balanced acc: 0.95065, best_iter: 217
Training until validation scores don't improve for 50 rounds
[100]	valid_0's multi_logloss: 0.173284
[200]	valid_0's multi_logloss: 0.169459
Early stopping, best iteration is:
[225]	valid_0's multi_logloss: 0.169303
Fold 1 balanced acc: 0.95120, best_iter: 225
Training until validation scores don't improve for 50 rounds
[100]	valid_0's multi_logloss: 0.179158
[200]	valid_0's multi_logloss: 0.175982
Early stopping, best iteration is:
[229]	valid_0's multi_logloss: 0.175919
Fold 2 balanced acc: 0.94884, best_iter: 229
Training until validation scores don't improve for 50 rounds
[100]	valid_0's multi_logloss: 0.177307
[200]	valid_0's multi_logloss: 0.173669
Early stopping, best iteration is:
[211]	valid_0's multi_logloss: 0.173619
F

In [6]:
print(f"Overall OOF balanced acc: {balanced_accuracy_score(y, oof.argmax(axis=1)):.5f}")

pred_labels = le.inverse_transform(test_preds.argmax(axis=1))
sub = pd.DataFrame({'id': test['id'], 'health_condition': pred_labels})
sub.to_csv('submission.csv', index=False)

Overall OOF balanced acc: 0.94955


In [7]:
oof_preds_labels = le.inverse_transform(oof.argmax(axis=1))
true_labels = le.inverse_transform(y)

print(classification_report(true_labels, oof_preds_labels, digits=4))

print("Confusion matrix (rows=true, cols=pred), order:", le.classes_)
print(confusion_matrix(true_labels, oof_preds_labels, labels=le.classes_))

              precision    recall  f1-score   support

     at-risk     0.9934    0.9363    0.9640    592561
         fit     0.7347    0.9487    0.8281     39803
   unhealthy     0.6936    0.9637    0.8067     57724

    accuracy                         0.9393    690088
   macro avg     0.8072    0.9496    0.8662    690088
weighted avg     0.9534    0.9393    0.9430    690088

Confusion matrix (rows=true, cols=pred), order: ['at-risk' 'fit' 'unhealthy']
[[554805  13402  24354]
 [  1827  37762    214]
 [  1863    235  55626]]


In [8]:
cat_features_idx = [X.columns.get_loc(c) for c in cat_cols]

X_cb = X.copy()
X_test_cb = X_test.copy()
for c in cat_cols:
    X_cb[c] = X_cb[c].astype(str).replace('nan', 'missing')
    X_test_cb[c] = X_test_cb[c].astype(str).replace('nan', 'missing')

oof_cb = np.zeros((len(X_cb), 3))
test_preds_cb = np.zeros((len(X_test_cb), 3))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_cb, y)):
    X_tr, X_val = X_cb.iloc[tr_idx], X_cb.iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    train_pool = Pool(X_tr, y_tr, cat_features=cat_features_idx)
    val_pool = Pool(X_val, y_val, cat_features=cat_features_idx)
    test_pool = Pool(X_test_cb, cat_features=cat_features_idx)

    model_cb = CatBoostClassifier(
        iterations=2000,
        learning_rate=0.05,
        depth=8,
        loss_function='MultiClass',
        eval_metric='TotalF1',
        auto_class_weights='Balanced',
        random_seed=42,
        early_stopping_rounds=50,
        verbose=100,
    )
    model_cb.fit(train_pool, eval_set=val_pool, use_best_model=True)

    oof_cb[val_idx] = model_cb.predict_proba(X_val)
    test_preds_cb += model_cb.predict_proba(test_pool) / n_splits

    fold_bal_acc = balanced_accuracy_score(y_val, oof_cb[val_idx].argmax(axis=1))
    print(f"CatBoost Fold {fold} balanced acc: {fold_bal_acc:.5f}")

print(f"CatBoost Overall OOF balanced acc: {balanced_accuracy_score(y, oof_cb.argmax(axis=1)):.5f}")

0:	learn: 0.9106249	test: 0.9115584	best: 0.9115584 (0)	total: 2.38s	remaining: 1h 19m 23s
100:	learn: 0.9476978	test: 0.9481576	best: 0.9481576 (100)	total: 3m 55s	remaining: 1h 13m 46s
200:	learn: 0.9491251	test: 0.9492143	best: 0.9492375 (199)	total: 7m 25s	remaining: 1h 6m 24s
300:	learn: 0.9498580	test: 0.9494313	best: 0.9494448 (262)	total: 11m 16s	remaining: 1h 3m 41s
400:	learn: 0.9506029	test: 0.9495707	best: 0.9496968 (379)	total: 15m 17s	remaining: 1h 57s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.9496967569
bestIteration = 379

Shrink model to first 380 iterations.
CatBoost Fold 0 balanced acc: 0.94965
0:	learn: 0.9101688	test: 0.9117216	best: 0.9117216 (0)	total: 2.29s	remaining: 1h 16m 23s
100:	learn: 0.9475718	test: 0.9495271	best: 0.9495661 (95)	total: 3m 58s	remaining: 1h 14m 52s
200:	learn: 0.9490285	test: 0.9504187	best: 0.9504187 (200)	total: 7m 50s	remaining: 1h 10m 9s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.9504

In [9]:
blend_oof = (oof + oof_cb) / 2
blend_test = (test_preds + test_preds_cb) / 2

print(f"Blend OOF balanced acc: {balanced_accuracy_score(y, blend_oof.argmax(axis=1)):.5f}")

pred_labels_blend = le.inverse_transform(blend_test.argmax(axis=1))
sub_blend = pd.DataFrame({'id': test['id'], 'health_condition': pred_labels_blend})
sub_blend.to_csv('submission_blend.csv', index=False)

Blend OOF balanced acc: 0.94959
